Nakdimon Dataset
This notebook make Hebrew fluers dataset with nikud

In [ ]:
# conda create -n nakdimon_env python=3.11 -y
# conda activate nakdimon_env
# pip install nakdimon
# conda install -n nakdimon_env ipykernel --update-deps --force-reinstall

#conda create -n nakdimon_env python=3.11 cudatoolkit=11.8.0 cudnn=8.9.2.26 -c conda-forge -y
# conda activate nakdimon_env
# pip install tensorflow==2.15.0 nakdimon ipykernel
# pip install tourch
# pip install torch
# pip install torchaudio
# pip install phonemizer
# pip install datasets
# pip install ipywidgets

In [ ]:
import os
import json
import torch
import torchaudio
import nakdimon
import time
from data.tokenizer import AudioTokenizer, TextTokenizer
import re
from datasets import concatenate_datasets, load_dataset
from tqdm import tqdm
import subprocess

In [18]:
# Downloading and Loading dataset 
print("Downloading and Loading the Fleurs dataset...")

try:
    # Train dataset
    print("Train dataset...")
    dataset_train_orig = load_dataset(
        "google/fleurs", 
        "he_il", 
        split="train", 
        trust_remote_code=True
    )
    
    print("\n--- Success! ---")
    print(f"Train Dataset loaded. Number of samples: {len(dataset_train_orig)}")
    
    # Check one sample to make sure audio and text are linked
    sample = dataset_train_orig[1388]
    print(f"Text: {sample['transcription']}")
    print(f"Audio array shape: {sample['audio']['array'].shape}")

except Exception as e:
    print(f"Error loading: {e}")
    print("If it fails, we will try to point it directly to the download folder.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/fleurs' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Train dataset...


Using the latest cached version of the dataset since google/fleurs couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'he_il' at /home/sukiennik/.cache/huggingface/datasets/google___fleurs/he_il/2.0.0/80cb68d1b4d319aefbd8ea302274d3950d95f6242f0742c1452d1545c80a2d5f (last modified on Tue Mar  3 11:00:04 2026).



--- Success! ---
Train Dataset loaded. Number of samples: 3242
Text: המחאה החלה בערך ב-11:00 זמן מקומי utcּ+1 בווייטהול מול הכניסה לרחוב דאונינג השמורה על ידי שוטרים המשכן הרשמי של ראש הממשלה
Audio array shape: (211200,)


In [26]:
    sample = dataset_train_orig[1065]
    print(f"Text: {sample['transcription']}")
    print(f"Audio array shape: {sample['audio']['array'].shape}")

Text: המחאה החלה בערך ב-11:00 זמן מקומי utcּ+1 בווייטהול מול הכניסה לרחוב דאונינג השמורה על ידי שוטרים המשכן הרשמי של ראש הממשלה
Audio array shape: (202560,)


In [ ]:
try:
    # Validation dataset
    print("Validation dataset...")
    dataset_val = load_dataset(
        "google/fleurs", 
        "he_il", 
        split="validation", 
        trust_remote_code=True
    )
    
    print("\n--- Success! ---")
    print(f"Validation Dataset loaded. Number of samples: {len(dataset_val)}")
    
    # Check one sample to make sure audio and text are linked
    sample = dataset_val[0]
    print(f"Text: {sample['transcription']}")
    print(f"Audio array shape: {sample['audio']['array'].shape}")

except Exception as e:
    print(f"Error loading: {e}")
    print("If it fails, we will try to point it directly to the download folder.")

In [ ]:
try:
    # Test dataset
    print("Test dataset...")
    dataset_test_orig = load_dataset(
        "google/fleurs", 
        "he_il", 
        split="test", 
        trust_remote_code=True
    )
    
    print("\n--- Success! ---")
    print(f"Test Dataset loaded. Number of samples: {len(dataset_test_orig)}")
    
    # Check one sample to make sure audio and text are linked
    sample = dataset_test_orig[0]
    print(f"Text: {sample['transcription']}")
    print(f"Audio array shape: {sample['audio']['array'].shape}")

except Exception as e:
    print(f"Error loading: {e}")
    print("If it fails, we will try to point it directly to the download folder.")

In [ ]:
# Re-balancing the dataset to maximize training data

# Split the Test into: 700 for Train, 91 for final Test
test_for_train = dataset_test_orig.select(range(700))
dataset_test = dataset_test_orig.select(range(700, len(dataset_test_orig)))

# Combine original Train with the extra Test samples
dataset_train = concatenate_datasets([dataset_train_orig, test_for_train])

print(f"New Train size: {len(dataset_train)}")
print(f"Final Test size: {len(dataset_test)}")

In [2]:
import nakdimon
print(dir(nakdimon))

['MAIN_MODEL', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'argparse', 'config', 'diacritize', 'diacritize_main', 'do_metrics', 'do_predict', 'do_run_test', 'do_server', 'do_train', 'logging', 'main', 'os', 'sys']


In [2]:
import os

# הדרך הבטוחה: הגדרת הנתיב רק עבור הסקריפט הזה
# זה לא משנה כלום במערכת ההפעלה שלך באופן קבוע
if 'CONDA_PREFIX' in os.environ:
    conda_lib_path = os.path.join(os.environ['CONDA_PREFIX'], 'lib')
    os.environ['LD_LIBRARY_PATH'] = conda_lib_path + ":" + os.environ.get('LD_LIBRARY_PATH', '')

import tensorflow as tf
import nakdimon

# עכשיו נבדוק
print("GPU list:", tf.config.list_physical_devices('GPU'))

2026-03-07 20:40:56.292114: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 20:40:56.414298: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-07 20:40:56.414367: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-07 20:40:56.421513: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-07 20:40:56.448214: I tensorflow/core/platform/cpu_feature_guar

GPU list: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2026-03-07 20:40:58.439864: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-07 20:40:58.440146: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-07 20:40:58.440167: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [3]:
import tensorflow as tf
print("Devices detected:", tf.config.list_physical_devices())
# You want to see 'GPU' in the output list

Devices detected: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
import tensorflow as tf

# Check if TensorFlow was built with CUDA (GPU support)
print("Built with CUDA:", tf.test.is_built_with_cuda())

# List all available physical devices
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ Success! Found {len(gpus)} GPU(s):")
    for gpu in gpus:
        print(f"  - {gpu}")
else:
    print("❌ GPU still not detected. Running on CPU.")

Built with CUDA: True
✅ Success! Found 1 GPU(s):
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [4]:
import nakdimon

# The main function in the package seems to be 'diacritize'
text = "שלום עולם"

# Run the diacritization process
# Note: This usually takes a few seconds on first run to load models
result = nakdimon.diacritize(text)

print(result)

2026-03-07 11:18:34.840895: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 11:18:34.894563: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-07 11:18:35.147910: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-07 11:18:35.147998: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-07 11:18:35.194420: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

1/1 [==============================] - 6s 6s/step
שָׁלוֹם עוֹלַם


In [5]:
text_tokenizer_he = TextTokenizer(backend="espeak", language="he")

In [6]:
# Test cases to check how the tokenizer handles diacritics
test_sentences = [
    "דגש",          # No diacritics
    "דָּגָשׁ",       # With Dagesh and Kamatz
    "ספר",          # Could be Sefer or Sfar
    "סֵפֶר",         # Sefer (Book)
    "סְפָר"          # Sfar (Border)
]

print("--- Phoneme Tokenization Test ---")
for text in test_sentences:
    # Running your specific tokenizer
    phonemes = text_tokenizer_he(text)[0]
    phonemes_str = " ".join(phonemes)
    print(f"Text: {text:10} | Phonemes: {phonemes_str}")

--- Phoneme Tokenization Test ---
Text: דגש        | Phonemes: d ɡ ʃ
Text: דָּגָשׁ    | Phonemes: d a ɡ a ʃ
Text: ספר        | Phonemes: s e f e ʁ
Text: סֵפֶר      | Phonemes: s e f e ʁ
Text: סְפָר      | Phonemes: s f a ʁ


In [7]:
# Final test for B/V, K/Kh, P/F distinction
test_pairs = [
    ("בַּיִת", "בּ"), # Bayit (B)
    ("גַּב", "ב"),    # Gav (V)
    ("כַּלְבָּה", "כּ"), # Kalba (K)
    ("מֶלֶךְ", "ך"),   # Melekh (Kh)
    ("פִּיל", "פּ"),   # Pil (P)
    ("קוף", "פ")     # Kof (F)
]

print("--- Begadkephat Phoneme Test ---")
for word, char in test_pairs:
    try:
        phonemes = text_tokenizer_he(word)[0]
        print(f"Word: {word:10} | Phonemes: {' '.join(phonemes)}")
    except Exception as e:
        print(f"Error: {e}")

--- Begadkephat Phoneme Test ---
Word: בַּיִת     | Phonemes: b a i t
Word: גַּב       | Phonemes: ɡ i m e l _ ( en ) h iː b ɹ uː d a ɡ ɛ ʃ ( he ) _ a _ v e t
Word: כַּלְבָּה  | Phonemes: k a l b a ʔ
Word: מֶלֶךְ     | Phonemes: m e l e χ
Word: פִּיל      | Phonemes: p i j l
Word: קוף        | Phonemes: k v f


In [28]:
# --- Configuration ---
manifest_dir = "./voicecraft_data_nakdimon/manifest"
# Define the files we want to process
splits = {
    "train": "train_manifest_filtered.jsonl",
    "val": "val_manifest_filtered.jsonl",
    "test": "test_manifest_filtered.jsonl"
}

# --- Helper Functions ---
def final_cleanup(text):
    """ Cleans text artifacts before diacritization """
    if not text: return ""
    return " ".join(text.replace('\\', '').replace('"', "'").split())

def clean_dagesh_for_phonemes(text):
    """ Removes problematic dagesh for the phonemizer """
    return text.replace('ּ', '')

# Ensure the 'tests' directory exists (Nakdimon CLI requirement)
if not os.path.exists('tests'):
    os.makedirs('tests')

for split_name, filename in splits.items():
    input_manifest = os.path.join(manifest_dir, filename)
    output_manifest = os.path.join(manifest_dir, f"{split_name}_manifest_nikud.jsonl")
    
    temp_raw = f"temp_{split_name}_raw.txt"
    temp_dotted = f"temp_{split_name}_dotted.txt"

    if not os.path.exists(input_manifest):
        print(f"⏩ Skipping {split_name}: File not found.")
        continue

    print(f"\n--- Processing {split_name.upper()} Split ---")
    
    # Step 1: Extract and clean text
    print(f"📦 Loading {filename}...")
    with open(input_manifest, 'r', encoding='utf-8') as f:
        items = [json.loads(line) for line in f]

    with open(temp_raw, "w", encoding="utf-8") as f:
        for item in items:
            f.write(final_cleanup(item['text']) + "\n")

    # Step 2: Run Nakdimon Batch Diacritization
    print(f"🪄 Running Nakdimon on {len(items)} sentences...")
    try:
        subprocess.run([
            "python", "-m", "nakdimon", "predict", 
            temp_raw, temp_dotted
        ], check=True)
    except Exception as e:
        print(f"❌ Error in Nakdimon for {split_name}: {e}")
        continue

    # Step 3: Rebuild the manifest with dotted text and new phonemes
    if os.path.exists(temp_dotted):
        with open(temp_dotted, "r", encoding="utf-8") as f:
            dotted_lines = f.read().splitlines()

        print(f"📝 Generating phonemes and saving to {output_manifest}...")
        with open(output_manifest, 'w', encoding='utf-8') as f_out:
            for item, dotted in tqdm(zip(items, dotted_lines), total=len(items)):
                final_text = clean_dagesh_for_phonemes(dotted)
                item['text'] = final_text
                
                # Update phonemes based on the new dotted text
                try:
                    phonemes_result = text_tokenizer_he(final_text)
                    item['phonemes'] = " ".join(phonemes_result[0])
                except:
                    pass # Keep original phonemes if tokenizer fails
                
                f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
        
        # Cleanup temp files for this split
        os.remove(temp_raw)
        os.remove(temp_dotted)
        print(f"✅ {split_name.upper()} is ready!")

print("\n🏁 ALL DONE! Your entire dataset is now diacritized and ready for training.")


--- Processing TRAIN Split ---
📦 Loading train_manifest_filtered.jsonl...
🪄 Running Nakdimon on 3632 sentences...


2026-03-08 00:28:07.322026: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-08 00:28:07.322167: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-08 00:28:07.329456: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-08 00:28:08.317149: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


2/2 [==============================] - 20s 5s/step
📝 Generating phonemes and saving to ./voicecraft_data_nakdimon/manifest/train_manifest_nikud.jsonl...


100%|██████████| 3632/3632 [00:00<00:00, 4161.80it/s]


✅ TRAIN is ready!

--- Processing VAL Split ---
📦 Loading val_manifest_filtered.jsonl...
🪄 Running Nakdimon on 294 sentences...


...

✅ VAL is ready!

--- Processing TEST Split ---
📦 Loading test_manifest_filtered.jsonl...
🪄 Running Nakdimon on 84 sentences...



2026-03-08 00:28:52.342725: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-08 00:28:52.342808: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-08 00:28:52.343690: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-08 00:28:52.954474: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


1/1 [==============================] - 8s 8s/step
📝 Generating phonemes and saving to ./voicecraft_data_nakdimon/manifest/test_manifest_nikud.jsonl...


100%|██████████| 84/84 [00:00<00:00, 3826.13it/s]

✅ TEST is ready!

🏁 ALL DONE! Your entire dataset is now diacritized and ready for training.


In [ ]:
def convert_jsonl_to_voicecraft_txt(jsonl_input_path, txt_output_path, phonemes_base_dir):
    """
    Converts a JSONL manifest into the specific TXT format required by VoiceCraft
    and creates individual phoneme files for each sample.
    
    jsonl_input_path: Path to the source .jsonl file
    txt_output_path: Path where the final .txt manifest will be saved
    phonemes_base_dir: Directory where individual .txt phoneme files will be created
    """
    # Create phonemes directory if it doesn't exist
    os.makedirs(phonemes_base_dir, exist_ok=True)
    
    print(f"Processing: {jsonl_input_path} -> {txt_output_path}")
    
    samples_processed = 0
    with open(jsonl_input_path, 'r', encoding='utf-8') as f_in, \
         open(txt_output_path, 'w', encoding='utf-8') as f_out:
        
        for i, line in enumerate(f_in):
            data = json.loads(line)
            
            # Extract ID from audio_filepath (e.g., 'sample_0')
            item_id = os.path.basename(data['audio_filepath']).replace(".wav", "")
            duration = data.get('duration_frames', 0)
            phonemes = data.get('phonemes', "")
            
            # Create the individual phoneme file (e.g., ./phonemes/sample_0.txt)
            phn_file_path = os.path.join(phonemes_base_dir, f"{item_id}.txt")
            with open(phn_file_path, "w", encoding="utf-8") as f_phn:
                f_phn.write(phonemes)
            
            # Write to the manifest .txt (Format: Index <TAB> ID <TAB> Duration)
            f_out.write(f"{i}\t{item_id}\t{duration}\n")
            samples_processed += 1
            
    print(f"Successfully converted {samples_processed} samples.")

# --- Usage Example ---
# Define paths and call the function for train and validation
base_manifest_dir = "./voicecraft_data/manifest"
phn_dir = "./voicecraft_data/phonemes"

# For Train
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "train_manifest.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "train.txt"),
    phonemes_base_dir=phn_dir
)

# For Validation
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "val_manifest.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "validation.txt"),
    phonemes_base_dir=phn_dir
)

# For Test
convert_jsonl_to_voicecraft_txt(
    jsonl_input_path=os.path.join(base_manifest_dir, "test_manifest.jsonl"),
    txt_output_path=os.path.join(base_manifest_dir, "test.txt"),
    phonemes_base_dir=phn_dir
)

In [ ]:
# 1. יצירת קובץ טקסט זמני עם כל המשפטים (אחד בכל שורה)
input_manifest = "./voicecraft_data_nakdimon/manifest/train_manifest.jsonl"
temp_raw = "all_texts_raw.txt"
temp_dotted = "all_texts_dotted.txt"

with open(input_manifest, 'r', encoding='utf-8') as f:
    items = [json.loads(line) for line in f]

with open(temp_raw, "w", encoding="utf-8") as f:
    for item in items:
        # ניקוי בסיסי לפני השליחה לנקדימון
        clean_text = final_cleanup(item['text'])
        f.write(clean_text + "\n")

# 2. יצירת תיקיית tests פיקטיבית כדי שנקדימון לא יקרוס
if not os.path.exists('tests'):
    os.makedirs('tests')

print(f"🚀 מריץ את נקדימון על {len(items)} משפטים בבת אחת...")

# 3. הרצת ה-CLI (זה אמור לקחת בערך דקה על ה-GPU)
try:
    subprocess.run([
        "python", "-m", "nakdimon", "predict", 
        temp_raw, temp_dotted
    ], check=True)
    print("✅ הניקוד הסתיים בהצלחה!")
except subprocess.CalledProcessError as e:
    print(f"❌ שגיאה בהרצת נקדימון: {e}")

# 4. קריאת התוצאות ועדכון המניפסט הסופי
if os.path.exists(temp_dotted):
    with open(temp_dotted, "r", encoding="utf-8") as f:
        dotted_lines = f.read().splitlines()

    output_manifest = "./voicecraft_data_nakdimon/manifest/train_manifest_fixed.jsonl"
    with open(output_manifest, 'w', encoding='utf-8') as f_out:
        for item, dotted in zip(items, dotted_lines):
            # ניקוי דגשים ועיבוד פונמות
            final_text = clean_dagesh_for_phonemes(dotted)
            item['text'] = final_text
            
            # יצירת פונמות (זה מהיר על ה-CPU)
            phonemes_result = text_tokenizer_he(final_text)
            item['phonemes'] = " ".join(phonemes_result[0])
            
            f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
    
    print(f"🏁 המניפסט המוכן נשמר ב: {output_manifest}")

🚀 מריץ את נקדימון על 3942 משפטים בבת אחת...


2026-03-07 21:37:32.882249: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-07 21:37:32.882303: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-07 21:37:32.883345: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-07 21:37:33.520853: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


2/2 [==============================] - 22s 6s/step
✅ הניקוד הסתיים בהצלחה!
🏁 המניפסט המוכן נשמר ב: ./voicecraft_data_nakdimon/manifest/train_manifest_fixed.jsonl


In [ ]:
# 1. יצירת קובץ טקסט זמני עם כל המשפטים (אחד בכל שורה)
input_manifest = "./voicecraft_data_nakdimon/manifest/train_manifest.jsonl"
temp_raw = "all_texts_raw.txt"
temp_dotted = "all_texts_dotted.txt"

with open(input_manifest, 'r', encoding='utf-8') as f:
    items = [json.loads(line) for line in f]

with open(temp_raw, "w", encoding="utf-8") as f:
    for item in items:
        # ניקוי בסיסי לפני השליחה לנקדימון
        clean_text = final_cleanup(item['text'])
        f.write(clean_text + "\n")

# 2. יצירת תיקיית tests פיקטיבית כדי שנקדימון לא יקרוס
if not os.path.exists('tests'):
    os.makedirs('tests')

print(f"🚀 מריץ את נקדימון על {len(items)} משפטים בבת אחת...")

# 3. הרצת ה-CLI (זה אמור לקחת בערך דקה על ה-GPU)
try:
    subprocess.run([
        "python", "-m", "nakdimon", "predict", 
        temp_raw, temp_dotted
    ], check=True)
    print("✅ הניקוד הסתיים בהצלחה!")
except subprocess.CalledProcessError as e:
    print(f"❌ שגיאה בהרצת נקדימון: {e}")

# 4. קריאת התוצאות ועדכון המניפסט הסופי
if os.path.exists(temp_dotted):
    with open(temp_dotted, "r", encoding="utf-8") as f:
        dotted_lines = f.read().splitlines()

    output_manifest = "./voicecraft_data_nakdimon/manifest/train_manifest_fixed.jsonl"
    with open(output_manifest, 'w', encoding='utf-8') as f_out:
        for item, dotted in zip(items, dotted_lines):
            # ניקוי דגשים ועיבוד פונמות
            final_text = clean_dagesh_for_phonemes(dotted)
            item['text'] = final_text
            
            # יצירת פונמות (זה מהיר על ה-CPU)
            phonemes_result = text_tokenizer_he(final_text)
            item['phonemes'] = " ".join(phonemes_result[0])
            
            f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
    
    print(f"🏁 המניפסט המוכן נשמר ב: {output_manifest}")

🚀 מריץ את נקדימון על 3942 משפטים בבת אחת...


2026-03-07 21:37:32.882249: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-07 21:37:32.882303: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-07 21:37:32.883345: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-07 21:37:33.520853: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


2/2 [==============================] - 22s 6s/step
✅ הניקוד הסתיים בהצלחה!
🏁 המניפסט המוכן נשמר ב: ./voicecraft_data_nakdimon/manifest/train_manifest_fixed.jsonl


In [13]:
# --- Configuration ---
INPUT_PATH = "./voicecraft_data_nakdimon/manifest/train_manifest.jsonl"
OUTPUT_PATH = "./voicecraft_data_nakdimon/manifest/train_manifest_fixed.jsonl"
BATCH_SIZE = 32  # GPU handles multiple sentences at once

# Final cleanup function (English capitalization, double quotes)
def final_cleanup(text):
    text = re.sub(r'\b([a-z])([a-z]+)\b', lambda m: m.group(1).upper() + m.group(2), text)
    text = text.replace('""', '"').replace('\\"', '"')
    text = re.sub(r'[\u0591-\u05C7]', '', text)
    return text

def clean_dagesh_for_phonemes(text):
    return re.sub(r'(?<![בכפ])\u05bc', '', text)

def run_fast_cli_processing():
    input_manifest = "./voicecraft_data_nakdimon/manifest/train_manifest.jsonl"
    output_manifest = "./voicecraft_data_nakdimon/manifest/train_manifest_fixed.jsonl"
    
    # 1. Read the current manifest
    with open(input_manifest, 'r', encoding='utf-8') as f:
        items = [json.loads(line) for line in f]
    
    # 2. Write raw texts to a temp file for Nakdimon CLI
    temp_input = "raw_texts_for_nakdimon.txt"
    with open(temp_input, "w", encoding="utf-8") as f:
        for item in items:
            f.write(final_cleanup(item['text']) + "\n")
    
    print(f"🚀 Running Nakdimon PREDICT on {len(items)} lines...")
    temp_output = "dotted_texts_from_nakdimon.txt"
    
    # 3. Execute Nakdimon CLI Predict
    try:
        # We call it as a module to ensure it uses your nakdimon_env
        subprocess.run([
            "python", "-m", "nakdimon", "predict", 
            temp_input, temp_output
        ], check=True)
    except Exception as e:
        print(f"❌ CLI Error: {e}")
        return

    # 4. Read the diacritized results
    with open(temp_output, "r", encoding="utf-8") as f:
        dotted_lines = f.read().splitlines()

    # 5. Safety check: make sure line counts match
    if len(dotted_lines) != len(items):
        print(f"⚠️ Warning: Line count mismatch! Input: {len(items)}, Output: {len(dotted_lines)}")
        # If mismatch, we use the smaller number to avoid crash
        limit = min(len(dotted_lines), len(items))
    else:
        limit = len(items)

    # 6. Rebuild manifest and update phonemes
    print("✍️ Updating phonemes and saving manifest...")
    with open(output_manifest, 'w', encoding='utf-8') as f_out:
        for i in range(limit):
            item = items[i]
            dotted = dotted_lines[i]
            
            final_text = clean_dagesh_for_phonemes(dotted)
            item['text'] = final_text
            
            # Regenerate phonemes (this part is fast CPU work)
            phonemes_result = text_tokenizer_he(final_text)
            item['phonemes'] = " ".join(phonemes_result[0])
            
            f_out.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"✅ SUCCESS! Final file is ready: {output_manifest}")

# Start the process
run_fast_cli_processing()

🚀 Running Nakdimon PREDICT on 3942 lines...
❌ CLI Error: Command '['python', '-m', 'nakdimon', 'predict', 'raw_texts_for_nakdimon.txt', 'dotted_texts_from_nakdimon.txt']' returned non-zero exit status 1.


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/sukiennik/miniconda3/envs/nakdimon_env/lib/python3.11/site-packages/nakdimon/__main__.py", line 4, in <module>
    nakdimon.main()
  File "/home/sukiennik/miniconda3/envs/nakdimon_env/lib/python3.11/site-packages/nakdimon/__init__.py", line 57, in main
    available_tests = [f'tests/{folder}' for folder in os.listdir('tests/')
                                                       ^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'tests/'


In [15]:
def final_cleanup(text):
    # Capitalize English words and fix double quotes
    text = re.sub(r'\b([a-z])([a-z]+)\b', lambda m: m.group(1).upper() + m.group(2), text)
    text = text.replace('""', '"').replace('\\"', '"')
    text = re.sub(r'[\u0591-\u05C7]', '', text)
    return text

def clean_dagesh_for_phonemes(text):
    # Keep dagesh only for Bet, Kaf, Pe to avoid phonemizer confusion
    return re.sub(r'(?<![בכפ])\u05bc', '', text)

def run_final_fix_v3():
    input_manifest = "./voicecraft_data_nakdimon/manifest/train_manifest.jsonl"
    output_manifest = "./voicecraft_data_nakdimon/manifest/train_manifest_fixed.jsonl"
    
    # 1. Load all data
    with open(input_manifest, 'r', encoding='utf-8') as f:
        items = [json.loads(line) for line in f]
    
    # 2. Prepare clean texts
    print(f"🧹 Cleaning {len(items)} texts...")
    raw_texts = [final_cleanup(item['text']) for item in items]
    
    # 3. Batch Diacritization (The GPU part)
    # Sending the WHOLE list at once is the fastest way in this library version
    print("🔥 Diacritizing everything in one GPU batch (this might take a minute)...")
    try:
        # We pass the list directly. If it fails, we'll fall back to a faster loop.
        dotted_results = nakdimon.diacritize(raw_texts)
    except Exception as e:
        print(f"⚠️ Batch failed: {e}. Falling back to list comprehension...")
        # If the library is stubborn, this loop is still faster than before 
        # because the model is already initialized in the background.
        dotted_results = [nakdimon.diacritize(t) for t in tqdm(raw_texts)]

    # 4. Save and generate phonemes
    print("✍️ Updating phonemes and saving results...")
    with open(output_manifest, 'w', encoding='utf-8') as f_out:
        for item, dotted in zip(items, dotted_results):
            # Ensure we have a string, not a list
            final_dotted = dotted[0] if isinstance(dotted, list) else dotted
            
            final_text = clean_dagesh_for_phonemes(final_dotted)
            item['text'] = final_text
            
            # Re-generate phonemes (Fast CPU task)
            try:
                phonemes_result = text_tokenizer_he(final_text)
                item['phonemes'] = " ".join(phonemes_result[0])
            except:
                item['phonemes'] = "" # Fallback for bad samples
                
            f_out.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"✅ FINISHED! Manifest saved to: {output_manifest}")

# Run it!
run_final_fix_v3()

🧹 Cleaning 3942 texts...
🔥 Diacritizing everything in one GPU batch (this might take a minute)...
⚠️ Batch failed: 'list' object has no attribute 'split'. Falling back to list comprehension...


  0%|          | 0/3944 [00:00<?, ?it/s]

1/1 [==============================] - 4s 4s/step


  0%|          | 1/3944 [00:04<4:53:36,  4.47s/it]

1/1 [==============================] - 4s 4s/step


...

1/1 [==============================] - 6s 6s/step


  8%|▊         | 327/3944 [32:44<6:17:13,  6.26s/it]

1/1 [==============================] - 6s 6s/step


  8%|▊         | 328/3944 [32:50<6:09:06,  6.12s/it]

KeyboardInterrupt: 

In [11]:
# --- Configuration ---
INPUT_PATH = "./voicecraft_data_nakdimon/manifest/train_manifest.jsonl"
OUTPUT_PATH = "./voicecraft_data_nakdimon/manifest/train_manifest_fixed.jsonl"
BATCH_SIZE = 32  # GPU handles multiple sentences at once

def final_cleanup(text):
    # Capitalize English and fix double quotes
    text = re.sub(r'\b([a-z])([a-z]+)\b', lambda m: m.group(1).upper() + m.group(2), text)
    text = text.replace('""', '"').replace('\\"', '"')
    text = re.sub(r'[\u0591-\u05C7]', '', text)
    return text

def clean_dagesh_for_phonemes(text):
    return re.sub(r'(?<![בכפ])\u05bc', '', text)

def run_fast_gpu_batch():
    with open(INPUT_PATH, 'r', encoding='utf-8') as f:
        all_lines = f.readlines()

    if os.path.exists(OUTPUT_PATH):
        os.remove(OUTPUT_PATH)
    
    # We'll use a larger batch to keep the GPU busy
    BATCH_SIZE = 64 
    print(f"🚀 High-Speed GPU Batching: {len(all_lines)} items, Batch Size: {BATCH_SIZE}")

    with open(OUTPUT_PATH, 'a', encoding='utf-8') as f_out:
        for i in tqdm(range(0, len(all_lines), BATCH_SIZE)):
            batch_lines = all_lines[i : i + BATCH_SIZE]
            batch_items = [json.loads(line) for line in batch_lines]
            
            # Prepare clean texts
            raw_texts = [final_cleanup(item['text']) for item in batch_items]
            
            try:
                # --- THIS IS THE FIX ---
                # Instead of diacritize(), we use the batch-friendly approach if available,
                # or we ensure the list is handled correctly.
                # If nakdimon.diacritize doesn't take a list, we wrap it:
                dotted_results = [nakdimon.diacritize(t) for t in raw_texts]
                
                for item, dotted in zip(batch_items, dotted_results):
                    # Nakdimon sometimes returns a list for a single string
                    if isinstance(dotted, list): dotted = dotted[0]
                    
                    final_text = clean_dagesh_for_phonemes(dotted)
                    item['text'] = final_text
                    
                    # Phonemes
                    phonemes_result = text_tokenizer_he(final_text)
                    item['phonemes'] = " ".join(phonemes_result[0])
                    
                    f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
                
                f_out.flush()
                
            except Exception as e:
                print(f"Error in batch starting at {i}: {e}")

run_fast_gpu_batch()

🚀 High-Speed GPU Batching: 3942 items, Batch Size: 64


  0%|          | 0/62 [00:00<?, ?it/s]

1/1 [==============================] - 7s 7s/step


  0%|          | 0/62 [04:20<?, ?it/s]


KeyboardInterrupt: 

In [10]:
def final_cleanup(text):
    # 1. Capitalize English words to encourage word-level phonemes instead of spelling
    text = re.sub(r'\b([a-z])([a-z]+)\b', lambda m: m.group(1).upper() + m.group(2), text)
    # 2. Fix double quotes issue (e.g., in Dr. abbreviation)
    text = text.replace('""', '"').replace('\\"', '"')
    # 3. Strip existing "dirty" niqqud/dagesh from raw text before Nakdimon
    text = re.sub(r'[\u0591-\u05C7]', '', text)
    return text

def clean_dagesh_for_phonemes(text):
    # Keep dagesh only for Bet, Kaf, Pe to avoid phonemizer confusion
    return re.sub(r'(?<![בכפ])\u05bc', '', text)

def process_everything():
    # Define paths
    target_dir = "./voicecraft_data_nakdimon"
    manifest_path = os.path.join(target_dir, "manifest/train_manifest.jsonl")
    
    updated_items = []

    # Open and read the manifest file
    with open(manifest_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            
            # Step 1: Preliminary cleaning
            clean_text = final_cleanup(item['text'])
            
            # Step 2: Apply Nakdimon diacritization
            dotted = nakdimon.diacritize(clean_text)
            
            # Step 3: Remove problematic dagesh marks
            final_text = clean_dagesh_for_phonemes(dotted)
            
            # Update the text field in the dictionary
            item['text'] = final_text
            
            # Step 4: Regenerate phonemes using the tokenizer
            try:
                # text_tokenizer_he returns ([phonemes], [ids])
                # We only need the phonemes list at index 0
                phonemes_result = text_tokenizer_he(final_text)
                item['phonemes'] = " ".join(phonemes_result[0])
            except Exception as e:
                print(f"Error processing phonemes for sample {item.get('sample_id')}: {e}")
            
            updated_items.append(item)

    # Step 5: Save the cleaned and diacritized manifest back to disk
    with open(manifest_path, 'w', encoding='utf-8') as f:
        for item in updated_items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
            
    print(f"✅ Processed {len(updated_items)} samples successfully.")

# Run the processing pipeline
process_everything()

1/1 [==============================] - 7s 7s/step


KeyboardInterrupt: 

In [11]:
def clean_problematic_dagesh(text):
    """
    Removes Dagesh from letters that cause the phonemizer to output English descriptions.
    We keep it for Bet, Kaf, and Pe to preserve B/V, K/Kh, P/F distinctions.
    """
    # Keep dagesh only for: ב, כ, פ
    # Remove it from any other character (like ג, ד, ת or others)
    return re.sub(r'(?<![בכפ])\u05bc', '', text)

def process_voicecraft_manifests_final(base_path, batch_size=100):
    """
    Processes all JSONL manifests with optimized diacritization and phonemization.
    """
    manifest_dir = os.path.join(base_path, "manifest")
    files = ["train_manifest.jsonl", "val_manifest.jsonl", "test_manifest.jsonl"]
    
    for filename in files:
        file_path = os.path.join(manifest_dir, filename)
        if not os.path.exists(file_path):
            continue
            
        print(f"\n🚀 Processing {filename}...")
        
        with open(file_path, 'r', encoding='utf-8') as f:
            entries = [json.loads(line) for line in f]
        
        updated_entries = []
        for i in range(0, len(entries), batch_size):
            batch = entries[i : i + batch_size]
            raw_texts = [entry['text'] for entry in batch]
            
            # 1. Batch Diacritization (Nakdimon)
            diacritized_list = nakdimon.diacritize("\n".join(raw_texts)).split("\n")
            
            for j, entry in enumerate(batch):
                # 2. Clean diacritized text to avoid phonemizer errors
                # This ensures letters like Gimel don't trigger "gimel with dagesh" in English
                safe_text = clean_problematic_dagesh(diacritized_list[j])
                entry['text'] = safe_text
                
                # 3. Update Phonemes (Should be clean now)
                try:
                    phonemes_list = text_tokenizer_he(safe_text)[0]
                    entry['phonemes'] = " ".join(phonemes_list)
                except Exception as e:
                    print(f"      Warning: Failed to phonemize sample {entry.get('sample_id')}: {e}")
                
            updated_entries.extend(batch)
            if (i + batch_size) % 500 == 0:
                print(f"   Progress: {min(i + batch_size, len(entries))} / {len(entries)}")

        # 4. Save updated manifest (Overwrite in the new directory)
        with open(file_path, 'w', encoding='utf-8') as f:
            for entry in updated_entries:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        
        print(f"✅ Finished updating {filename}")

# --- EXECUTION ---
target_dir = "./voicecraft_data_nakdimon"

if os.path.exists(target_dir):
    process_voicecraft_manifests_final(target_dir)
    print("\n🎉 ALL DONE! Your diacritized dataset is ready for training.")
else:
    print("❌ Target directory not found. Please copy the data first.")


🚀 Processing train_manifest.jsonl...
1/1 [==============================] - 4s 4s/step
   Progress: 500 / 3942
1/1 [==============================] - 4s 4s/step
   Progress: 1000 / 3942


AssertionError: 6034, ['מקומי', 'utcּ+1', 'בווייטהול'], ּ, ['מ', 'ק', 'ו', 'מ', 'י', 'u', 't', 'c', 'דגש\\שורוק', '+', '1', 'ב', 'ו', 'ו', 'י', 'י', 'ט', 'ה', 'ו', 'ל']

In [ ]:
# --- Helper Function for Batch Diacritization ---
def diacritize_batch(manifest_entries):
    """
    Process a list of entries and add diacritics to text in one bulk operation.
    This significantly improves performance on CPU.
    """
    if not manifest_entries:
        return manifest_entries
    
    # Extract raw transcriptions for batch processing
    raw_texts = [entry['text'] for entry in manifest_entries]
    
    # Use newline as a separator for Nakdimon processing
    combined_text = "\n".join(raw_texts)
    
    # Perform diacritization (Inference)
    diacritized_full = nakdimon.diacritize(combined_text)
    
    # Split the results back into a list
    diacritized_list = diacritized_full.split("\n")
    
    # Update entries with the new diacritized text
    for i, entry in enumerate(manifest_entries):
        entry['text_diacritized'] = diacritized_list[i]
        
    return manifest_entries

# --- Main Processing Function ---
def process_split_with_diacritics(dataset, split_name, start_id, batch_size=100):
    split_manifest = []
    current_id = start_id
    
    print(f"\n--- Processing {split_name} split ({len(dataset)} samples) ---")
    
    # Loop through the dataset in batches for efficiency
    for i in range(0, len(dataset), batch_size):
        batch_entries = []
        # Define the subset for the current batch
        subset_indices = range(i, min(i + batch_size, len(dataset)))
        
        for idx in subset_indices:
            try:
                sample = dataset[idx]
                raw_text = sample['transcription']
                
                # Audio processing and tensor conversion
                audio_array = torch.tensor(sample['audio']['array']).unsqueeze(0)
                sr = sample['audio']['sampling_rate']
                
                # Calculate duration in frames (VoiceCraft specific)
                num_audio_samples = audio_array.shape[1]
                duration_frames = int((num_audio_samples / sr) * 50)
                
                # Path configuration
                wav_filename = f"sample_{current_id}.wav"
                save_path = os.path.join(wav_root_dir, split_name, wav_filename)
                manifest_audio_path = f"{split_name}/{wav_filename}"
                
                # Save audio to physical disk
                torchaudio.save(save_path, audio_array, sr)
                
                # Create initial entry structure
                batch_entries.append({
                    "audio_filepath": manifest_audio_path,
                    "text": raw_text,
                    "duration_frames": duration_frames,
                    "sample_id": current_id
                })
                current_id += 1
                
            except Exception as e:
                print(f"Error in {split_name} at index {idx}: {e}")

        # --- Batch Diacritization Step ---
        # Update all texts in the current batch with Nakdimon
        batch_entries = diacritize_batch(batch_entries)
        
        # Optional: Add phoneme processing here using 'text_diacritized'
        for entry in batch_entries:
            # phonemes_list = text_tokenizer_he(entry['text_diacritized'])[0]
            # entry['phonemes'] = " ".join(phonemes_list)
            pass

        split_manifest.extend(batch_entries)
        print(f"[{split_name}] Processed {len(split_manifest)} samples...")
            
    return split_manifest, current_id

# --- Execution ---
# global_counter = 0
# train_manifest, global_counter = process_split_with_diacritics(dataset_train, "train", global_counter)

In [ ]:
# --- Main Manifest Creation Loop ---
# --- Configuration ---
base_dir = "./voicecraft_data_nakdimon"
manifest_dir = os.path.join(base_dir, "manifest")
wav_root_dir = "./fleurs_hebrew/voicecraft_samples"

# Create directories
os.makedirs(manifest_dir, exist_ok=True)
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(wav_root_dir, split), exist_ok=True)

# Helper function to process a dataset split
def process_split(dataset, split_name, start_id):
    split_manifest = []
    current_id = start_id
    
    print(f"\n--- Processing {split_name} split ({len(dataset)} samples) ---")
    
    for i in range(len(dataset)):
        try:
            sample = dataset[i]
            raw_text = sample['transcription']
            
            # Audio processing
            audio_array = torch.tensor(sample['audio']['array']).unsqueeze(0)
            sr = sample['audio']['sampling_rate']
            
            # VoiceCraft duration calculation
            num_audio_samples = audio_array.shape[1]
            duration_frames = int((num_audio_samples / sr) * 50)
            
            # Naming with global continuous ID
            wav_filename = f"sample_{current_id}.wav"
            # Physical path for saving
            save_path = os.path.join(wav_root_dir, split_name, wav_filename)
            # Relative path for manifest (Kaggle-friendly)
            manifest_audio_path = f"{split_name}/{wav_filename}"
            
            # Save WAV
            torchaudio.save(save_path, audio_array, sr)
            
            # Get Phonemes (Assuming text_tokenizer_he is defined)
            phonemes_list = text_tokenizer_he(raw_text)[0]
            phonemes_str = " ".join(phonemes_list)
            
            # Build entry
            split_manifest.append({
                "audio_filepath": manifest_audio_path,
                "text": raw_text,
                "phonemes": phonemes_str,
                "duration_frames": duration_frames,
                "sample_id": current_id
            })
            
            if i % 100 == 0:
                print(f"[{split_name}] Processed {i} samples... (Global ID: {current_id})")
            
            current_id += 1 # Increment global counter
            
        except Exception as e:
            print(f"Error in {split_name} at index {i}: {e}")
            
    return split_manifest, current_id

# --- Execution ---

global_counter = 0

# 1. Process Train
train_manifest, global_counter = process_split(dataset_train, "train", global_counter)

# 2. Process Validation
val_manifest, global_counter = process_split(dataset_val, "val", global_counter)

# 3. Process Test
test_manifest, global_counter = process_split(dataset_test, "test", global_counter)

# --- Saving Individual Manifests ---
# Saving separate JSONL files for each split
splits_data = {
    "train_manifest.jsonl": train_manifest,
    "val_manifest.jsonl": val_manifest,
    "test_manifest.jsonl": test_manifest
}

for filename, data in splits_data.items():
    path = os.path.join(manifest_dir, filename)
    with open(path, "w", encoding="utf-8") as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"\nSuccessfully processed {global_counter} total samples.")
print(f"Manifests saved in: {manifest_dir}")